In [2]:
import polars as pl

# Load lazily to reduce RAM usage
df = pl.read_parquet("../data/malaysia_transactions.parquet")

In [3]:
print("Shape:", df.shape)
print("Columns:", df.columns)

Shape: (11998251, 19)
Columns: ['trxn_id', 'date_time', 'ofi_acct_number', 'ofi_bank_code', 'ofi_entity_id', 'ofi_entity_type', 'rfi_acct_number', 'rfi_bank_code', 'rfi_entity_id', 'rfi_entity_type', 'trxn_amount', 'trxn_channel', 'trxn_type', 'service_code', 'service_name', 'ofi_entity_name', 'rfi_entity_name', 'ofi_participant_type', 'rfi_participant_type']


### null count

In [4]:
print("🧯 Null count per column:")
display(df.null_count())

🧯 Null count per column:


trxn_id,date_time,ofi_acct_number,ofi_bank_code,ofi_entity_id,ofi_entity_type,rfi_acct_number,rfi_bank_code,rfi_entity_id,rfi_entity_type,trxn_amount,trxn_channel,trxn_type,service_code,service_name,ofi_entity_name,rfi_entity_name,ofi_participant_type,rfi_participant_type
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,438281,665434,442985,442985,464000,665434,0,0,665434,0,11364827,0,442985,0,0


### Is date_time sorted?

In [5]:
df["date_time"].is_sorted()

False

### Check if same ofi_entity_name → multiple ofi_acct_number

In [6]:
df.select(["ofi_entity_name", "ofi_acct_number"]) \
  .unique() \
  .group_by("ofi_entity_name") \
  .agg(pl.len().alias("num_accounts")) \
  .sort("num_accounts", descending=True) \
  .head(10) 

ofi_entity_name,num_accounts
str,u32
"""Unknown Entity""",398613
"""Ong Ahmad""",5456
"""Nurul binti Omar""",5451
"""Chong Wong""",5446
"""Li Hua Pillai""",5439
"""Priya Ahmad""",5431
"""Yusuf Menon""",5423
"""Ismail Ng""",5421
"""Ismail Ong""",5414


### Check if same entity id have multiple accounts

In [7]:
df.select(["ofi_entity_id", "ofi_acct_number"]) \
  .unique() \
  .group_by("ofi_entity_id") \
  .agg(pl.len().alias("num_accounts")) \
  .sort("num_accounts", descending=True) \
  .head(10) 

ofi_entity_id,num_accounts
str,u32
null,398613
"""671021-21-7041""",23
"""976578-I""",18
"""741023-02-5900""",4
"""790425-10-6073""",4
"""731227-08-6584""",4
"""770118-10-5695""",4
"""951022-14-7655""",4
"""570408-56-6830""",4


#### Posibilities null entity IDs
1. 🏢 Business/Corporate accounts (entity_id not required)
2. 🔧 Data processing errors (missing entity resolution)
3. ⚠️ System accounts (internal/technical accounts)
4. 🚨 Unverified accounts (incomplete KYC processes)

In [9]:
# Get top 3 entity IDs with highest number of accounts (excluding null)
top_entities = df.select(["ofi_entity_id", "ofi_acct_number"]) \
    .unique() \
    .group_by("ofi_entity_id") \
    .agg(pl.len().alias("num_accounts")) \
    .filter(pl.col("ofi_entity_id").is_not_null()) \
    .sort("num_accounts", descending=True) \
    .head(3) \
    .select("ofi_entity_id").to_series().to_list()

print(f"Top 3 entity IDs: {top_entities}")

# Show 5 rows with ALL columns for each entity
for entity_id in top_entities:
    print(f"\n🔍 ENTITY ID: {entity_id}")
    print("="*50)
    
    entity_data = df.filter(pl.col("ofi_entity_id") == entity_id).head(5)
    display(entity_data)

Top 3 entity IDs: ['671021-21-7041', '976578-I', '951022-14-7655']

🔍 ENTITY ID: 671021-21-7041


trxn_id,date_time,ofi_acct_number,ofi_bank_code,ofi_entity_id,ofi_entity_type,rfi_acct_number,rfi_bank_code,rfi_entity_id,rfi_entity_type,trxn_amount,trxn_channel,trxn_type,service_code,service_name,ofi_entity_name,rfi_entity_name,ofi_participant_type,rfi_participant_type
str,datetime[ns],str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str
"""23c27100-1f91-4fd8-a0c4-4e55df…",2025-06-28 18:55:02,"""FPB954779174350""","""First Penang Bank""","""671021-21-7041""",null,null,null,null,null,179.0,"""SAN""",null,"""None""","""Withdrawal""","""Ibrahim Reddy""",null,"""Unknown""","""Cash"""
"""521e36f6-4e2c-4638-9b1c-735ff2…",2025-06-17 01:33:44,"""FSB176566551813""","""First Selangor Bank""","""671021-21-7041""",null,null,null,null,null,179.0,"""SAN""",null,"""None""","""Withdrawal""","""Ibrahim Reddy""",null,"""Unknown""","""Cash"""
"""ff574ab4-ca13-4cb9-a2a8-b3ed42…",2025-06-04 07:23:05,"""FPB954779174350""","""First Penang Bank""","""671021-21-7041""",null,null,null,null,null,179.0,"""SAN""",null,"""None""","""Withdrawal""","""Ibrahim Reddy""",null,"""Unknown""","""Cash"""
"""39657f18-e95b-490f-8025-b72eee…",2025-06-21 10:30:10,"""FSB176566551813""","""First Selangor Bank""","""671021-21-7041""",null,null,null,null,null,900.0,"""SAN""",null,"""None""","""Withdrawal""","""Ibrahim Reddy""",null,"""Unknown""","""Cash"""
"""80afcf98-62c2-412c-ac17-9789c4…",2025-06-12 04:49:06,"""MCB969147130490""","""Malaysia Commercial Bank""","""671021-21-7041""",null,null,null,null,null,900.0,"""SAN""",null,"""None""","""Withdrawal""","""Ibrahim Reddy""",null,"""Unknown""","""Cash"""



🔍 ENTITY ID: 976578-I


trxn_id,date_time,ofi_acct_number,ofi_bank_code,ofi_entity_id,ofi_entity_type,rfi_acct_number,rfi_bank_code,rfi_entity_id,rfi_entity_type,trxn_amount,trxn_channel,trxn_type,service_code,service_name,ofi_entity_name,rfi_entity_name,ofi_participant_type,rfi_participant_type
str,datetime[ns],str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str
"""1508f211-5640-42a3-974f-f8acf2…",2025-06-26 20:43:29,"""FJB973462415036""","""First Johor Bank""","""976578-I""","""Personal""","""SB211389420980""","""Selangor Bank""","""684224-L""","""Business""",130.25,"""MyDebit""","""Card Payment""","""9999""",null,"""Prime Services Corp (949)""","""Royal International Commerce I…","""Bank/Wallet""","""Merchant/Acquirer"""
"""81d81938-1bb8-43e4-8436-ff8676…",2025-06-25 17:55:22,"""PTB463917611402""","""Penang Trust Bank""","""976578-I""","""Personal""","""BS793483278866""","""Bank of Sarawak""","""TR2228111-Y""","""Business""",41.85,"""MyDebit""","""Card Payment""","""9999""",null,"""Prime Services Corp (949)""","""Ultra Malaysia Systems Corp (8…","""Bank/Wallet""","""Merchant/Acquirer"""
"""05a88232-fd33-4820-9db9-c5fc6d…",2025-06-04 19:28:25,"""SB211389420980""","""Selangor Bank""","""976578-I""","""Personal""","""BS793483278866""","""Bank of Sarawak""","""200005046598""","""Business""",41.85,"""MyDebit""","""Card Payment""","""9999""",null,"""Prime Services Corp (949)""","""Smart Financial Co (741)""","""Bank/Wallet""","""Merchant/Acquirer"""
"""b3b5c449-702c-48f9-bfe5-1711ea…",2025-06-10 03:54:43,"""PB376967105320""","""Perak Bank""","""976578-I""","""Personal""","""BJ991281364892""","""Bank of Johor""","""201206266418""","""Business""",41.85,"""MyDebit""","""Card Payment""","""9999""",null,"""Prime Services Corp (949)""","""Royal International Group Sdn …","""Bank/Wallet""","""Merchant/Acquirer"""
"""6ceec5d1-a077-4e8e-83f8-eb2536…",2025-06-01 18:13:25,"""BP198107986150""","""Bank of Perak""","""976578-I""","""Personal""","""SB211389420980""","""Selangor Bank""","""952990-Q""","""Business""",66.25,"""MyDebit""","""Card Payment""","""9999""",null,"""Prime Services Corp (949)""","""Imperial Construction Sdn Bhd …","""Bank/Wallet""","""Merchant/Acquirer"""



🔍 ENTITY ID: 951022-14-7655


trxn_id,date_time,ofi_acct_number,ofi_bank_code,ofi_entity_id,ofi_entity_type,rfi_acct_number,rfi_bank_code,rfi_entity_id,rfi_entity_type,trxn_amount,trxn_channel,trxn_type,service_code,service_name,ofi_entity_name,rfi_entity_name,ofi_participant_type,rfi_participant_type
str,datetime[ns],str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str
"""17ef8cfb-bf5e-4d50-94d6-e4ec48…",2025-06-16 15:09:06,"""JTB072219376350""","""Johor Trust Bank""","""951022-14-7655""","""Business""","""KIB959392261918""","""Kedah Islamic Bank""","""640329-07-5629""","""Business""",53.48,"""RPP""","""QR Payment""","""070""",null,"""Khadijah Kumar""","""Hassan Yusof""","""Bank/Wallet""","""Merchant/Acquirer"""
"""4aa8081c-70f7-4168-b233-b21916…",2025-06-28 13:49:38,"""SB896592484767""","""Selangor Bank""","""951022-14-7655""","""Personal""","""STB126866372697""","""Sarawak Trust Bank""","""200006185608""","""Business""",53.48,"""FPX""","""Online Payment""","""9999""",null,"""Khadijah Kumar""","""Elite Marketing Co (614)""","""Bank/Wallet""","""Merchant/Acquirer"""
"""e7577f31-7766-42f3-ae2d-5bc7f2…",2025-06-06 19:42:57,"""SB912492660379""","""Selangor Bank""","""951022-14-7655""","""Personal""","""STB126866372697""","""Sarawak Trust Bank""","""KL2103136-T""","""Business""",42.59,"""FPX""","""Online Payment""","""9999""",null,"""Meera Kumar""","""Alpha Southeast Development Lt…","""Bank/Wallet""","""Merchant/Acquirer"""
"""5184e5d1-3fc8-4232-a524-9d9245…",2025-06-08 14:19:56,"""SB897690721394""","""Selangor Bank""","""951022-14-7655""","""Personal""","""BS793483278866""","""Bank of Sarawak""","""202012124570""","""Business""",7.0,"""MyDebit""","""Card Payment""","""9999""",null,"""Deepa Hassan""","""Pioneer Properties Ltd (162)""","""Bank/Wallet""","""Merchant/Acquirer"""
"""69676c16-48cc-43c6-ad43-79d67d…",2025-06-23 23:25:26,"""SB896592484767""","""Selangor Bank""","""951022-14-7655""","""Business""","""JTB355666805048""","""Johor Trust Bank""","""683902-Z""","""Business""",19.3,"""RPP""","""QR Payment""","""030""",null,"""Khadijah Kumar""","""Alpha Pacific Manufacturing Lt…","""Bank/Wallet""","""Merchant/Acquirer"""


### Check if one ofi_entity_id is used by multiple ofi_entity_name

In [27]:
df.select(["ofi_entity_id", "ofi_entity_name"]) \
  .unique() \
  .group_by("ofi_entity_id") \
  .agg(pl.len().alias("num_entities")) \
  .sort("num_entities", descending=False) \
  .head(1000) 



ofi_entity_id,num_entities
str,u32
"""981008-08-5741""",1
"""PK8100502-E""",1
"""691006-07-6101""",1
"""750625-31-7784""",1
"""950505-03-6049""",1
…,…
"""900516-02-6187""",1
"""680910-08-5393""",1
"""960626-14-7386""",1


In [10]:
# Get top 5 entity IDs with multiple entity names
top_multi_name_entities = df.select(["ofi_entity_id", "ofi_entity_name"]) \
  .unique() \
  .group_by("ofi_entity_id") \
  .agg(pl.len().alias("num_entities")) \
  .sort("num_entities", descending=True) \
  .head(5) \
  .select("ofi_entity_id").to_series().to_list()

print(f"Top 5 entity IDs with multiple names: {top_multi_name_entities}")

# Show the entity names for each of these entity IDs
for entity_id in top_multi_name_entities:
    print(f"\n🔍 ENTITY ID: {entity_id}")
    
    entity_names = df.filter(pl.col("ofi_entity_id") == entity_id) \
        .select("ofi_entity_name").unique().to_series().to_list()
    
    print(f"   Entity names: {entity_names}")

Top 5 entity IDs with multiple names: ['760913-02-7009', '935379-E', '918695-D', '790712-07-6026', '541120-01-6755']

🔍 ENTITY ID: 760913-02-7009
   Entity names: ['Tan Pillai', 'Siti Teo', 'Aishah Ong', 'Ng Wong']

🔍 ENTITY ID: 935379-E
   Entity names: ['Elite Asia Retail Bhd (712)', 'Prime International Financial LLC (400)', 'Supreme Southeast Systems Ltd (715)', 'Pioneer Resources Co (397)']

🔍 ENTITY ID: 918695-D
   Entity names: ['Global Services Pte Ltd (245)', 'Digital Retail Co (192)', 'Digital Corporation Bhd (257)']

🔍 ENTITY ID: 790712-07-6026
   Entity names: ['Arjun Yusof', 'Raj bin Ali', 'Matthew Robertson']

🔍 ENTITY ID: 541120-01-6755
   Entity names: ['Zainab Tan', 'Li Hua binti Omar', 'Ahmad Nair']


### Check if same ofi_acct_number → multiple ofi_entity_name

In [7]:
df.select(["ofi_entity_name", "ofi_acct_number"]) \
  .unique() \
  .group_by("ofi_acct_number") \
  .agg(pl.len().alias("num_names")) \
  .sort("num_names", descending=True) \
  .head(10)

ofi_acct_number,num_names
str,u32
"""SB017321485862""",2
"""SB012773041259""",2
"""BP860206152897""",2
"""SB172599859656""",2
"""SB179726784038""",2
"""FJB405547364530""",1
"""BP911360016953""",1
"""SB475981872650""",1
"""BJ539309666764""",1


In [8]:
# Get accounts with multiple names
suspicious_accounts_df = df.select(["ofi_entity_name", "ofi_acct_number"]) \
  .unique() \
  .group_by("ofi_acct_number") \
  .agg(pl.len().alias("num_names")) \
  .filter(pl.col("num_names") > 1) \
  .sort("num_names", descending=True) \
  .head(5)

# Get the account numbers
suspicious_accounts = suspicious_accounts_df.select("ofi_acct_number").to_series().to_list()

# Show ALL columns for these suspicious accounts
suspicious_data = df.filter(pl.col("ofi_acct_number").is_in(suspicious_accounts)) \
                   .sort(["ofi_acct_number", "date_time"])

print(f"Complete data for {len(suspicious_accounts)} suspicious accounts:")
print(f"Total transactions: {suspicious_data.height}")
display(suspicious_data)

Complete data for 5 suspicious accounts:
Total transactions: 13


trxn_id,date_time,ofi_acct_number,ofi_bank_code,ofi_entity_id,ofi_entity_type,rfi_acct_number,rfi_bank_code,rfi_entity_id,rfi_entity_type,trxn_amount,trxn_channel,trxn_type,service_code,service_name,ofi_entity_name,rfi_entity_name,ofi_participant_type,rfi_participant_type
str,datetime[ns],str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str
"""b286d488-681d-40cc-9456-9404a7…",2025-06-02 19:42:16,"""BP860206152897""","""Bank of Perak""","""530321-07-7899""","""Personal""","""SB532682180150""","""Selangor Bank""","""200110046761""","""Business""",2664.68,"""FPX""","""Online Payment""","""9999""",null,"""Zainab Singh""","""Royal Investment Bhd (726)""","""Bank/Wallet""","""Merchant/Acquirer"""
"""dccff7fe-557d-4cd4-8a44-750fca…",2025-06-08 08:02:31,"""BP860206152897""","""Bank of Perak""","""571230-08-5158""","""Personal""","""BS793483278866""","""Bank of Sarawak""","""201311185596""","""Business""",7.0,"""MyDebit""","""Card Payment""","""9999""",null,"""Ng Hassan""","""Royal Financial LLC (324)""","""Bank/Wallet""","""Merchant/Acquirer"""
"""c0b22b84-54a7-47ba-925e-6acb82…",2025-06-13 22:24:45,"""BP860206152897""","""Bank of Perak""","""530321-07-7899""","""Personal""","""SB482052095024""","""Selangor Bank""","""831119-01-5421""","""Personal""",41.31,"""RPP""","""Online Transfer""","""040""",null,"""Zainab Singh""","""Ismail Ahmad""","""Bank/Wallet""","""Bank/Wallet"""
"""b5122861-29a0-4b2f-895c-09db19…",2025-06-25 09:14:13,"""BP860206152897""","""Bank of Perak""","""571230-08-5158""","""Personal""","""BS793483278866""","""Bank of Sarawak""","""201311185596""","""Business""",7.0,"""MyDebit""","""Card Payment""","""9999""",null,"""Ng Hassan""","""Royal Financial LLC (324)""","""Bank/Wallet""","""Merchant/Acquirer"""
"""a3f8edec-d5cd-41de-9161-a866dd…",2025-06-20 17:38:19,"""SB012773041259""","""Selangor Bank""","""910226-14-5419""","""Personal""","""BS793483278866""","""Bank of Sarawak""","""PK4125541-C""","""Business""",11.0,"""MyDebit""","""Card Payment""","""9999""",null,"""Jasmine Bautista""","""Innovative Solutions Ltd (582)""","""Bank/Wallet""","""Merchant/Acquirer"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""1870ea59-51ac-471d-abf4-5df576…",2025-06-02 01:48:55,"""SB172599859656""","""Selangor Bank""","""PG7678177-Z""","""Business""","""SB010488806171""","""Selangor Bank""","""KD4558104-Y""","""Business""",7.0,"""RPP""","""QR Payment""","""030""",null,"""Excellence Worldwide Consultin…","""Pioneer Solutions Corp (203)""","""Bank/Wallet""","""Merchant/Acquirer"""
"""597bfa1b-c27d-4e6f-9051-56898e…",2025-06-13 18:39:06,"""SB172599859656""","""Selangor Bank""","""890108-08-5807""","""Personal""","""SB445278841077""","""Selangor Bank""","""920322-08-7405""","""Personal""",7.0,"""RPP""","""Online Transfer""","""040""",null,"""Khadijah Menon""","""Dr. Christopher Mann""","""Bank/Wallet""","""Bank/Wallet"""
"""a0755b5a-4caf-40f4-b2ad-5946c5…",2025-06-24 00:31:02,"""SB172599859656""","""Selangor Bank""","""PG7678177-Z""","""Business""","""JTB840990127518""","""Johor Trust Bank""","""677438-U""","""Business""",7.0,"""RPP""","""QR Payment""","""030""",null,"""Excellence Worldwide Consultin…","""Digital Development Ltd (725)""","""Bank/Wallet""","""Merchant/Acquirer"""


### Check if one ofi_acct_number is owned by multiple ofi_entity_id

In [13]:
df.select(["ofi_entity_id", "ofi_acct_number"]) \
  .unique() \
  .group_by("ofi_acct_number") \
  .agg(pl.len().alias("num_id")) \
  .sort("num_id", descending=True) \
  .head(10)

ofi_acct_number,num_id
str,u32
"""SB017321485862""",2
"""BP860206152897""",2
"""SB012773041259""",2
"""SB172599859656""",2
"""SB179726784038""",2
"""JTB411712497451""",1
"""FSB166478007421""",1
"""FJB855249436132""",1
"""SB510256870702""",1


### Check if same rfi_entity_name → multiple rfi_acct_number

In [15]:
df.select(["rfi_entity_name", "rfi_acct_number"]) \
  .unique() \
  .group_by("rfi_entity_name") \
  .agg(pl.len().alias("num_accounts")) \
  .sort("num_accounts", descending=True) \
  .head(10)

rfi_entity_name,num_accounts
str,u32
"""Unknown Entity""",15810
"""Siti Hassan""",2673
"""Muhammad Yusof""",2649
"""Khadijah Singh""",2641
"""Ahmad Ibrahim""",2627
"""Siti Devi""",2621
"""Wei Ming Ahmad""",2619
"""Wei Ming Nair""",2614
"""Ali Ibrahim""",2612


### Check if same rfi_entity_id → multiple rfi_acct_number

In [14]:
df.select(["rfi_entity_id", "rfi_acct_number"]) \
  .unique() \
  .group_by("rfi_entity_id") \
  .agg(pl.len().alias("num_accounts")) \
  .sort("num_accounts", descending=True) \
  .head(10)

rfi_entity_id,num_accounts
str,u32
null,15811
"""976578-I""",5
"""914542-O""",3
"""889784-K""",3
"""808562-M""",3
"""781016-01-6703""",3
"""843040-P""",3
"""630304-T""",3
"""700624-14-5150""",3


### Check if same rfi_acct_number → multiple rfi_entity_name

In [22]:
df.select(["rfi_entity_name", "rfi_acct_number"]) \
  .unique() \
  .group_by("rfi_acct_number") \
  .agg(pl.len().alias("num_names")) \
  .sort("num_names", descending=True) \
  .head(1000)

rfi_acct_number,num_names
str,u32
"""STB126866372697""",452738
"""BP177380680284""",125473
"""PF032790211749""",124812
"""PF493628200973""",90792
"""SB339885003146""",86680
…,…
"""BP713645992824""",3
"""BP351610547559""",3
"""BJ041915695195""",3


### Check if same rfi_acct_number → multiple rfi_entity_id

In [28]:
df.select(["rfi_entity_id", "rfi_acct_number"]) \
  .unique() \
  .group_by("rfi_acct_number") \
  .agg(pl.len().alias("num_id")) \
  .sort("num_id", descending=False) \
  .head(1000)

rfi_acct_number,num_id
str,u32
"""SB954551348910""",1
"""JTB838106763639""",1
"""PB656432942963""",1
"""PIB660927218107""",1
"""JCB805232713310""",1
…,…
"""FSB106214340337""",1
"""SB709272051852""",1
"""SB679453949322""",1


### Check if one rfi_entity_id is used by multiple rfi_entity_name

In [25]:
df.select(["rfi_entity_id", "rfi_entity_name"]) \
  .unique() \
  .group_by("rfi_entity_id") \
  .agg(pl.len().alias("num_entities")) \
  .sort("num_entities", descending=True) \
  .head(10) 

rfi_entity_id,num_entities
str,u32
"""137065-G""",4
"""592486-L""",3
"""900705-W""",3
"""651971-N""",3
"""658984-S""",3
"""101516-Y""",3
"""818617-K""",3
"""289799-K""",3
"""116053-L""",3


### Clean multiple name to one id, multiple id to one acct

In [29]:
print("🧹 CREATING CLEAN DATASET")
print("=" * 50)

# Step 1: Identify clean OFI (sender) data
print("🔍 Step 1: Cleaning OFI (Sender) data...")

# Remove entity_ids used by multiple entity_names
clean_ofi_entity_ids = df.select(["ofi_entity_id", "ofi_entity_name"]) \
    .unique() \
    .group_by("ofi_entity_id") \
    .agg(pl.len().alias("num_names")) \
    .filter(pl.col("num_names") == 1) \
    .select("ofi_entity_id")

print(f"Clean OFI entity IDs: {clean_ofi_entity_ids.height:,}")

# Remove account_numbers used by multiple entity_ids
clean_ofi_accounts = df.select(["ofi_entity_id", "ofi_acct_number"]) \
    .unique() \
    .group_by("ofi_acct_number") \
    .agg(pl.len().alias("num_ids")) \
    .filter(pl.col("num_ids") == 1) \
    .select("ofi_acct_number")

print(f"Clean OFI accounts: {clean_ofi_accounts.height:,}")

# Step 2: Identify clean RFI (receiver) data
print("\n🔍 Step 2: Cleaning RFI (Receiver) data...")

# Remove entity_ids used by multiple entity_names
clean_rfi_entity_ids = df.select(["rfi_entity_id", "rfi_entity_name"]) \
    .unique() \
    .group_by("rfi_entity_id") \
    .agg(pl.len().alias("num_names")) \
    .filter(pl.col("num_names") == 1) \
    .select("rfi_entity_id")

print(f"Clean RFI entity IDs: {clean_rfi_entity_ids.height:,}")

# Remove account_numbers used by multiple entity_ids
clean_rfi_accounts = df.select(["rfi_entity_id", "rfi_acct_number"]) \
    .unique() \
    .group_by("rfi_acct_number") \
    .agg(pl.len().alias("num_ids")) \
    .filter(pl.col("num_ids") == 1) \
    .select("rfi_acct_number")

print(f"Clean RFI accounts: {clean_rfi_accounts.height:,}")

# Step 3: Create the clean dataset
print("\n🔧 Step 3: Creating clean dataset...")

# Filter original data to keep only clean records
clean_df = df.join(clean_ofi_entity_ids, on="ofi_entity_id", how="inner") \
           .join(clean_ofi_accounts, on="ofi_acct_number", how="inner") \
           .join(clean_rfi_entity_ids, on="rfi_entity_id", how="inner") \
           .join(clean_rfi_accounts, on="rfi_acct_number", how="inner")

print(f"\n📊 CLEANING RESULTS:")
print(f"Original dataset: {df.height:,} transactions")
print(f"Clean dataset: {clean_df.height:,} transactions")
print(f"Data retention: {(clean_df.height / df.height) * 100:.1f}%")

# Step 4: Verify data quality
print("\n✅ Step 4: Verifying data quality...")

# Check entity_id to entity_name consistency
ofi_id_name_check = clean_df.select(["ofi_entity_id", "ofi_entity_name"]) \
    .unique() \
    .group_by("ofi_entity_id") \
    .agg(pl.len().alias("num_names")) \
    .filter(pl.col("num_names") > 1)

rfi_id_name_check = clean_df.select(["rfi_entity_id", "rfi_entity_name"]) \
    .unique() \
    .group_by("rfi_entity_id") \
    .agg(pl.len().alias("num_names")) \
    .filter(pl.col("num_names") > 1)

print(f"OFI entity_ids with multiple names: {ofi_id_name_check.height}")
print(f"RFI entity_ids with multiple names: {rfi_id_name_check.height}")

# Check account to entity_id consistency
ofi_acct_id_check = clean_df.select(["ofi_entity_id", "ofi_acct_number"]) \
    .unique() \
    .group_by("ofi_acct_number") \
    .agg(pl.len().alias("num_ids")) \
    .filter(pl.col("num_ids") > 1)

rfi_acct_id_check = clean_df.select(["rfi_entity_id", "rfi_acct_number"]) \
    .unique() \
    .group_by("rfi_acct_number") \
    .agg(pl.len().alias("num_ids")) \
    .filter(pl.col("num_ids") > 1)

print(f"OFI accounts with multiple entity_ids: {ofi_acct_id_check.height}")
print(f"RFI accounts with multiple entity_ids: {rfi_acct_id_check.height}")

# Step 5: Update account intersection analysis
print("\n🔄 Step 5: Updated account intersection analysis...")

clean_ofi_accounts_set = set(clean_df.select("ofi_acct_number").unique().to_series().to_list())
clean_rfi_accounts_set = set(clean_df.select("rfi_acct_number").unique().to_series().to_list())
clean_intersection = clean_ofi_accounts_set & clean_rfi_accounts_set

print(f"Clean sender accounts: {len(clean_ofi_accounts_set):,}")
print(f"Clean receiver accounts: {len(clean_rfi_accounts_set):,}")
print(f"Clean intersecting accounts: {len(clean_intersection):,}")
print(f"Intersection rate: {(len(clean_intersection) / len(clean_ofi_accounts_set)) * 100:.1f}%")

🧹 CREATING CLEAN DATASET
🔍 Step 1: Cleaning OFI (Sender) data...
Clean OFI entity IDs: 6,888,619
Clean OFI accounts: 7,469,045

🔍 Step 2: Cleaning RFI (Receiver) data...
Clean RFI entity IDs: 4,635,583
Clean RFI accounts: 2,873,100

🔧 Step 3: Creating clean dataset...

📊 CLEANING RESULTS:
Original dataset: 11,998,251 transactions
Clean dataset: 7,733,973 transactions
Data retention: 64.5%

✅ Step 4: Verifying data quality...
OFI entity_ids with multiple names: 0
RFI entity_ids with multiple names: 0
OFI accounts with multiple entity_ids: 0
RFI accounts with multiple entity_ids: 0

🔄 Step 5: Updated account intersection analysis...
Clean sender accounts: 4,985,134
Clean receiver accounts: 2,815,280
Clean intersecting accounts: 516,145
Intersection rate: 10.4%


In [28]:
ofi_accounts = df.select("ofi_acct_number").unique()
rfi_accounts = df.select("rfi_acct_number").unique()

# Convert to sets for intersection
ofi_set = set(ofi_accounts["ofi_acct_number"].to_list())
rfi_set = set(rfi_accounts["rfi_acct_number"].to_list())

intersection = ofi_set & rfi_set

print("Intersecting accounts:", len(intersection))
print("Total sender (ofi) accounts:", len(ofi_set))
print("Total receiver (rfi) accounts:", len(rfi_set))

Intersecting accounts: 615021
Total sender (ofi) accounts: 7469050
Total receiver (rfi) accounts: 2875835


In [ ]:
edges = df.select([
    pl.col("ofi_acct_number").alias("src"),
    pl.col("rfi_acct_number").alias("dst"),
    pl.col("date_time")  # optional, but keep for now
])

edges_sorted = edges.sort("date_time")

In [1]:
# TODO: EDA: Is the channel and type useful? If no remove in cleaning stage.
# TODO: Cleaning: Remove columns like name and service code/name
# TODO: Create graph, determine existence of cycles, etc.

### Check if channel is useful